# AGN kNN-CDF at fixed number density

Notebook 01 selected the top 10% of BHs by luminosity, so the number of AGN
per simulation varied, and that variation had to be regressed out afterwards
(`remove_abundance`). That run found `n_AGN` correlates with `Omega_m` at
**Spearman rho = 0.98** -- almost a deterministic relation.

That's a problem, because the kNN-CDF depends on number density through

$$\mathrm{CDF}_k(r) = 1 - \sum_{j<k} \frac{\lambda^j e^{-\lambda}}{j!}, \qquad \lambda = n\,\tfrac{4}{3}\pi r^3$$

which is violently nonlinear in $n$, while `remove_abundance` only removes a
trend *linear* in $\log_{10} n$. Whatever nonlinear density dependence is left
over leaks into the sensitivity of any parameter correlated with $n$ -- i.e.
straight into `Omega_m`. So in notebook 01 we cannot cleanly separate
"`Omega_m` changes how AGN cluster" from "`Omega_m` changes how many AGN there are".

**Fix:** select exactly the *N brightest* AGN in every simulation. Number
density is then identical across the suite by construction, no abundance
regression is needed at all, and any surviving parameter response is
clustering. The cost: simulations with fewer than N eligible BHs must be
dropped, which turns whole-sim retention into the bias to watch instead.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt

from src import config
from src.pipeline import run_suite, survey_eligible_counts
from src.params import load_params, align_to_params
from src.selection_bias import diagnose_retention_bias
from src.sensitivity import sensitivity_table, summary_dataframe, infer_layout
from src.plotting import plot_scale_grid, plot_sensitivity_bar

SIM_PATH = config.SIM_PATH
PARAMS_FILE = config.PARAMS_FILE
OUTPUT_DIR = config.OUTPUT_DIR

## 1. How many AGN are available? (choosing N)

`survey_eligible_counts` just counts BHs above the mass floor with usable
luminosities -- no kd-trees, so it's fast. N is a direct trade-off:

- **larger N** -> denser tracer sample, smaller typical kNN distances, signal
  pushed to smaller r -- but more simulations dropped
- **smaller N** -> nearly every simulation retained, but a sparser sample and
  a noisier CDF

Taking a low percentile of the distribution keeps the density as high as
possible while dropping only a few percent of the suite.

In [ ]:
counts = survey_eligible_counts(sim_path=SIM_PATH)

print(counts["n_eligible"].describe())

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(counts["n_eligible"], bins=50, color="#1a1a2e")
ax.set_xlabel("eligible BHs per simulation")
ax.set_ylabel("simulations")
ax.spines[["top", "right"]].set_visible(False)

for pct in (1, 5, 10, 25):
    print(f"{pct:>2}th percentile: {np.percentile(counts['n_eligible'], pct):.0f}")

In [ ]:
# Drop ~5% of simulations; override by hand after looking at the histogram above.
N_TARGET = int(np.percentile(counts["n_eligible"], 5))

n_dropped = int((counts["n_eligible"] < N_TARGET).sum())
print(f"N_TARGET = {N_TARGET}")
print(f"would drop {n_dropped} / {len(counts)} simulations")
print(f"mean AGN separation ~ {config.BOXSIZE / N_TARGET ** (1/3):.2f} Mpc/h")

## 2. Generate fixed-N summaries

Set `GENERATE = False` on re-runs once the `.npz` exists.

In [ ]:
GENERATE = True

npz_path = f"{OUTPUT_DIR}/agn_knn_snap{config.SNAP}_M{config.MASS_CUT:.0e}_n{N_TARGET}.npz"

if GENERATE:
    result = run_suite(n_target=N_TARGET, sim_path=SIM_PATH, output_dir=OUTPUT_DIR)
else:
    data = np.load(npz_path, allow_pickle=True)
    result = {k: data[k] for k in ("sim_ids", "summaries", "nbh", "rgrid", "kvals")}

sim_ids = result["sim_ids"]
summaries = result["summaries"]
nbh = result["nbh"]
rgrid = result["rgrid"]
kvals = result["kvals"]
n_k, n_r = infer_layout(kvals, rgrid)

# The guarantee this whole notebook rests on:
assert len(np.unique(nbh)) == 1, f"tracer count is not constant: {np.unique(nbh)}"
print(f"{len(sim_ids)} simulations retained, {nbh[0]} AGN each, summary shape {summaries.shape}")

## 3. Parameters, alignment, and (no) abundance removal

`remove_abundance` is deliberately **not** called here: `nbh` is constant, so
there is no abundance trend left to remove. All that's needed is mean-centring
each bin -- and even that is cosmetic, since the quartile *difference* is
invariant to a constant per-bin offset.

In [ ]:
theta_all = load_params(PARAMS_FILE)
theta = align_to_params(sim_ids, theta_all)

residuals = summaries - summaries.mean(axis=0)

assert list(theta.index) == list(sim_ids)
assert residuals.shape[0] == len(theta)

## 4. Retention bias -- now the diagnostic that matters

Fixed-N trades one bias for another. The `nbh` confound is gone by
construction, but simulations too sparse to supply N AGN were dropped, and
those are preferentially the low-`Omega_m` ones. If retention correlates
strongly with a parameter, the retained suite no longer spans that
parameter's full prior range and its sensitivity is measured over a narrower
baseline (biased low), so check this before reading section 5.

In [ ]:
all_ids = theta_all.index.to_numpy()
retention_diag = diagnose_retention_bias(sim_ids, all_ids, theta_all, params=config.ALL_PARAMS)
display(retention_diag)

# How much of each parameter's prior range survives?
for p in config.ALL_PARAMS:
    full = theta_all[p]
    kept = theta[p]
    print(f"{p:>8}: full [{full.min():.3f}, {full.max():.3f}]  "
          f"retained [{kept.min():.3f}, {kept.max():.3f}]")

## 5. Sensitivity at fixed density

In [ ]:
table = sensitivity_table(
    residuals, theta,
    n_k=n_k, rgrid=rgrid, kvals=kvals,
    params=config.ALL_PARAMS,
    n_boot=2000, n_null=2000,
)

summary_df = summary_dataframe(table)
summary_df.sort_values("R_obs", ascending=False)

In [ ]:
fig, axes = plot_scale_grid(table, params=config.COSMO_PARAMS)
fig.suptitle("Cosmological parameters (fixed density)", y=1.02)

fig, axes = plot_scale_grid(table, params=config.ASTRO_PARAMS)
fig.suptitle("Astrophysical (feedback) parameters (fixed density)", y=1.02)

fig, ax = plot_sensitivity_bar(summary_df, config.COSMO_PARAMS, config.ASTRO_PARAMS)

## 6. Where the CDF actually has information

The response curves in section 5 are flat at small r and flat again at large
r. That isn't physics -- it's that the CDF is pinned at 0 below the smallest
AGN separations and saturated at 1 above the largest. Overplotting the mean
CDF and its slope shows how much of the apparent "scale dependence" is just
the envelope of where the statistic has any dynamic range at all.

If the response tracks `dCDF/dr`, the parameter is shifting the CDF sideways
and the signal is essentially one number, not a scale-dependent shape.

In [ ]:
mean_cdf = summaries.mean(axis=0).reshape(n_k, n_r)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

for ki in range(n_k):
    axes[0].plot(rgrid, mean_cdf[ki], label=f"k={kvals[ki]}")
    axes[1].plot(rgrid, np.gradient(mean_cdf[ki], np.log10(rgrid)), label=f"k={kvals[ki]}")

axes[0].set_ylabel("mean kNN-CDF")
axes[1].set_ylabel(r"$d\,\mathrm{CDF}/d\log r$")
for ax in axes:
    ax.set_xscale("log")
    ax.set_xlabel(r"$r\ [\mathrm{Mpc}/h]$")
    ax.axvline(config.BOXSIZE / 4, color="0.5", ls=":", lw=1)
    ax.legend(fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
print("dotted line = L/4; beyond it the 25 Mpc/h periodic box limits the measurement")

## 7. Does fixing the density change the answer?

The point of the whole exercise. If `Omega_m`'s response survives at fixed
density, it's genuine clustering. If it collapses, notebook 01 was largely
measuring abundance leaking through an imperfect linear correction.

In [ ]:
import pandas as pd

pct = int(round(config.TOP_FRACTION * 100))
old_path = f"{OUTPUT_DIR}/agn_knn_snap{config.SNAP}_M{config.MASS_CUT:.0e}_top{pct}.npz"

comparison = summary_df.set_index("parameter")[["R_obs", "q_value"]].add_suffix("_fixedN")

try:
    from src.abundance import remove_abundance
    from src.selection_bias import diagnose_nbh_confound, residualize_theta_on_nbh

    old = np.load(old_path, allow_pickle=True)
    old_theta = align_to_params(old["sim_ids"], theta_all)
    old_resid = remove_abundance(old["summaries"], old["nbh"])

    nbh_diag = diagnose_nbh_confound(old["nbh"], old_theta, params=config.ALL_PARAMS)
    confounded = nbh_diag.loc[nbh_diag["bias_flag"], "parameter"].tolist()
    old_theta_corr = (residualize_theta_on_nbh(old_theta, old["nbh"], params=confounded)
                      if confounded else old_theta)

    old_n_k, old_n_r = infer_layout(old["kvals"], old["rgrid"])
    old_table = sensitivity_table(
        old_resid, old_theta_corr,
        n_k=old_n_k, rgrid=old["rgrid"], kvals=old["kvals"],
        params=config.ALL_PARAMS, n_boot=2000, n_null=2000,
    )
    old_df = summary_dataframe(old_table).set_index("parameter")
    comparison = comparison.join(old_df[["R_obs", "q_value"]].add_suffix("_top10"))
    comparison["ratio"] = comparison["R_obs_fixedN"] / comparison["R_obs_top10"]
except FileNotFoundError:
    print(f"No top-fraction run at {old_path}; showing fixed-N only.")

comparison.sort_values("R_obs_fixedN", ascending=False)

## Next

With density fixed, the remaining structural limits are the **LH set itself**
(all 6 parameters vary at once, so each response is marginalised over the
other five, and every simulation has a different random seed) and the lack of
a **noise normalisation** (R is a signal with no error bar attached to it).
Both have direct fixes in CAMELS:

- **CV set** (27 sims, fiducial parameters, different seeds) measures the
  cosmic-variance floor -- the scatter R would show with *no* parameter
  variation at all. That converts R into an interpretable S/N.
- **1P set** (one parameter varied at a time, seed held fixed) gives clean
  derivatives dCDF/dp with no marginalisation and no seed noise.
- Then the **covariance** of the residuals across sims turns those
  derivatives into a Fisher matrix, whose off-diagonals *are* the
  parameter-degeneracy structure.